In [1]:
import pandas as pd
from backtesting import Backtest, Strategy
import math
from vnstock3 import Vnstock
import talib as ta

RSI_PERIOD = 14
RSI_OVERSOLD = 30
RSI_OVERBOUGHT = 70
BOLL_STD = 2
BOLL_PERIOD = 15

In [2]:
def calculate_first_mondays(dates):
        if not isinstance(dates, pd.DatetimeIndex):
            dates = pd.DatetimeIndex(dates)
        dates_series = pd.Series(dates, index=dates)
        mondays = dates_series[dates_series.dt.dayofweek == 0]
        first_mondays = mondays.groupby([mondays.dt.year, mondays.dt.month]).first()
        return set(first_mondays)
     
class DCA(Strategy):
    average_monthly_income_vnd = 1000
    investment_percentage = 0.10  # Percentage of income to invest
    fund = 0

    def init(self):
        close = self.data.Close
        # Calculate RSI
        self.rsi = self.I(ta.RSI, close, timeperiod=RSI_PERIOD)
        # Calculate Bollinger Bands
        self.boll_high = self.I(ta.BBANDS, close, timeperiod=BOLL_PERIOD, nbdevup=BOLL_STD, nbdevdn=BOLL_STD)[0]
        self.boll_low = self.I(ta.BBANDS, close, timeperiod=BOLL_PERIOD, nbdevup=BOLL_STD, nbdevdn=BOLL_STD)[2]
        self.first_mondays = calculate_first_mondays(self.data.index)
    def next(self):
        today = self.data.index[-1]
        self.data.Close[-1] = self.data.Close[-1] / 10
        if today in self.first_mondays:
            self.fund += self.average_monthly_income_vnd * self.investment_percentage

        # Check for buy signal
        if (self.data.Close[-1] <= self.boll_low[-1] and self.rsi[-1] <= RSI_OVERSOLD):
            share_price = self.data.Close[-1]
            shares_to_buy = self.fund // share_price
            shares_to_buy = (shares_to_buy // 100) * 100
            if shares_to_buy > 0:
                self.buy(size=shares_to_buy)
                self.fund -= share_price * shares_to_buy
                #print(f"Buy executed at {self.data.index[-1]} with {shares_to_buy} shares at price {share_price}, total price {share_price * shares_to_buy}")

                
def run_backtest(stock_symbol):
    # Fetch stock data
    stock_data = Vnstock().stock(symbol=stock_symbol,source='VCI').quote.history(start='2019-01-01', end='2024-01-04')
    stock_data = stock_data.rename(columns={"open": "Open", "high": "High", "low": "Low", "close": "Close", "volume": "Volume"})
    stock_data.set_index('time', inplace=True)
    stock_data.index = pd.to_datetime(stock_data.index)

    # Merge USD/VND data
    stock_data.index = stock_data.index.normalize()
    stock_data = stock_data.dropna()

    # Run the backtest
    bt = Backtest(
        stock_data,
        DCA,
        trade_on_close=True,
    )
    stats = bt.run()
    # bt.plot(filename=f'{stock_symbol}')
    
    # Calculate investment details
    trades = stats["_trades"]
    price_paid = trades["Size"] * trades["EntryPrice"]
    total_invested = price_paid.sum()

    current_shares = trades["Size"].sum()
    current_equity = current_shares * stock_data.Close.iloc[-1]

    print(f"Results for {stock_symbol}:")
    print(trades)
    print("Total investment:", total_invested)
    print("Current Shares:", current_shares)
    print("Current Equity:", current_equity)
    print("RoR:", ((current_equity - total_invested) / total_invested)*100)
    print("-" * 50)


# List of stock symbols
stock_symbols = ['FPT', 'MWG', 'E1VFVN30']

# Run backtest for each stock
for symbol in stock_symbols:
    run_backtest(symbol)

2024-12-23 21:06:08 - vnstock3.common.data.data_explorer - WARNING - Thông tin niêm yết & giao dịch sẽ được truy xuất từ TCBS


Results for FPT:
   Size  EntryBar  ExitBar  EntryPrice  ExitPrice     PnL  ReturnPct  \
0   100      1079     1251       5.647      8.259   261.2   0.462546   
1   100       940     1251       5.392      8.259   286.7   0.531714   
2   100       932     1251       5.866      8.259   239.3   0.407944   
3   300       757     1251       5.324      8.259   880.5   0.551277   
4   200       389     1251       2.168      8.259  1218.2   2.809502   
5   100       297     1251       2.015      8.259   624.4   3.098759   
6   500       266     1251       2.202      8.259  3028.5   2.750681   

   EntryTime   ExitTime  Duration  
0 2023-04-28 2024-01-03  250 days  
1 2022-10-07 2024-01-03  453 days  
2 2022-09-27 2024-01-03  463 days  
3 2022-01-11 2024-01-03  722 days  
4 2020-07-27 2024-01-03 1255 days  
5 2020-03-16 2024-01-03 1388 days  
6 2020-01-31 2024-01-03 1433 days  
Total investment: 5023.799999999999
Current Shares: 1400
Current Equity: 11670.4
RoR: 132.3022413312632
--------------

2024-12-23 21:06:09 - vnstock3.common.data.data_explorer - WARNING - Thông tin niêm yết & giao dịch sẽ được truy xuất từ TCBS
2024-12-23 21:06:11 - vnstock3.common.data.data_explorer - WARNING - Thông tin niêm yết & giao dịch sẽ được truy xuất từ TCBS


Results for MWG:
   Size  EntryBar  ExitBar  EntryPrice  ExitPrice    PnL  ReturnPct  \
0   200      1198     1251       4.266      4.286    4.0   0.004688   
1   100       966     1251       3.995      4.286   29.1   0.072841   
2   100       883     1251       5.994      4.286 -170.8  -0.284952   
3   300       837     1251       6.107      4.286 -546.3  -0.298182   
4   200       389     1251       2.358      4.286  385.6   0.817642   
5   100       292     1251       3.068      4.286  121.8   0.397001   
6   200       221     1251       3.631      4.286  131.0   0.180391   
7   100        68     1251       2.570      4.286  171.6   0.667704   

   EntryTime   ExitTime  Duration  
0 2023-10-19 2024-01-03   76 days  
1 2022-11-14 2024-01-03  415 days  
2 2022-07-18 2024-01-03  534 days  
3 2022-05-13 2024-01-03  600 days  
4 2020-07-27 2024-01-03 1255 days  
5 2020-03-09 2024-01-03 1395 days  
6 2019-11-21 2024-01-03 1504 days  
7 2019-04-16 2024-01-03 1723 days  
Total investment: 5